In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils
import als

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda
INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch11_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    num_recs = 100


config = Config()

In [4]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, do_ordinal_encoder_transform=True, drop_pad=True)
X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
    config,
    X_train,
    X_val,
    X_val_seq,
    ordinal_encoder,
    return_only_val=False,
    truncate_val=False,
    return_just_splitted=True,
)
pad = ordinal_encoder.unknown_value

CPU times: user 17min 29s, sys: 2min 46s, total: 20min 15s
Wall time: 4min 46s


In [5]:
train_val = pl.concat([X_train, X_val_seq[["user", "aid"]]])

In [6]:
data = train_val.to_pandas()[config.col_aid].tolist()
del train_val, X_train
gc.collect()

199

In [7]:
%%time

model_als = als.train_als_model(data=data, n_items=pad, iterations=50, factors=140, alpha=40.0, use_gpu=True, calculate_training_loss=True)

  0%|          | 0/50 [00:00<?, ?it/s]

CPU times: user 10min 31s, sys: 7.16 s, total: 10min 38s
Wall time: 10min 38s


In [8]:
val_seq_data = X_val_seq.to_pandas()[config.col_aid].tolist()

In [9]:
%%time

recommender = als.AlsRecommender(implicit_model=model_als)
recs = recommender.recommend(val_seq_data, n_recs=config.num_recs)

5it [00:28,  5.73s/it]


CPU times: user 50.4 s, sys: 2.9 s, total: 53.3 s
Wall time: 53.2 s


In [10]:
df_idx = recs.to_pandas()[["index"]].drop_duplicates().reset_index(drop=True)
df_idx["user"] = X_val_seq.to_pandas()["user"].astype(int)
df_idx = pl.from_pandas(df_idx)
recs = recs.join(df_idx, on="index")
recs.head()

index,item_id,als_score,als_rank,user
i64,i64,f64,i64,i64
0,831461,0.049393,1,13194112
0,838682,0.049363,2,13194112
0,289315,0.040964,3,13194112
0,807039,0.038434,4,13194112
0,270942,0.032819,5,13194112


In [11]:
%%time

recs = recs.to_pandas()
recs['item_id'] = ordinal_encoder.inverse_transform(
        recs["item_id"].values.reshape(-1, 1)
    ).reshape(1, -1)[0]
recs = recs.dropna()
recs = pl.from_pandas(recs)

CPU times: user 6.45 s, sys: 2.13 s, total: 8.58 s
Wall time: 7.58 s


In [12]:
%%time

recs.write_parquet(config.cur_model_output_path + f'cur_als_df_val_preds_{config.num_recs}.parquet')

CPU times: user 4.23 s, sys: 1.23 s, total: 5.46 s
Wall time: 4.91 s


In [13]:
recs.head()

index,item_id,als_score,als_rank,user
i64,i64,f64,i64,i64
0,1615582,0.049393,1,13194112
0,1629608,0.049363,2,13194112
0,563117,0.040964,3,13194112
0,1568011,0.038434,4,13194112
0,527209,0.032819,5,13194112


In [14]:
X_val_seq.head()

user,aid,train_seq
i64,list[i64],list[i64]
13194112,"[859226, 866443]","[859226, 866443]"
13057536,"[369809, 620054, ... 369809]","[369809, 620054, ... 369809]"
13580352,"[306782, 358166, 929474]","[306782, 358166, 929474]"
12921216,"[504173, 644435, ... 644435]","[504173, 644435, ... 644435]"
13595840,[320434],[320434]


In [15]:
X_val.filter(pl.col("user") == 14378816).head()

user,target,train_seq
i64,list[i64],list[i64]
14378816,"[257075, 1010867, 1578459]","[789219, 634920, ... 651794]"


In [16]:
recs = (
    recs.groupby(["index"])
    .agg(
        [
            pl.col("item_id").list(),
        ]
    )
    .sort("index")
    .to_pandas()
)

In [17]:
recs["user"] = X_val_seq.to_pandas()["user"]
recs.head()

,index,item_id,user
0,0,"[1615582, 1629608, 563117, 1568011, 527209, 44...",13194112
1,1,"[1695413, 878643, 1533580, 1468610, 1667230, 3...",13057536
2,2,"[799923, 1281615, 1116621, 384195, 1817803, 17...",13580352
3,3,"[479970, 738987, 319580, 122983, 1739400, 1139...",12921216
4,4,"[405663, 1524585, 479970, 521986, 496713, 1032...",13595840


In [18]:
%%time

recs["item_id"] = recs["item_id"].progress_apply(lambda x: utils.truncate_list(list(x), from_start=True, max_len=20))

  0%|          | 0/461865 [00:00<?, ?it/s]

CPU times: user 4.21 s, sys: 560 ms, total: 4.77 s
Wall time: 4.75 s


In [19]:
recs = recs.merge(X_val[["user", "target"]].to_pandas())

In [20]:
recs.head()

,index,item_id,user,target
0,0,"[1615582, 1629608, 563117, 1568011, 527209, 44...",13194112,"[1420545, 1669669, 1137693]"
1,1,"[1695413, 878643, 1533580, 1468610, 1667230, 3...",13057536,"[982447, 719005, 38838]"
2,2,"[799923, 1281615, 1116621, 384195, 1817803, 17...",13580352,"[688858, 1817953, 1817953]"
3,3,"[479970, 738987, 319580, 122983, 1739400, 1139...",12921216,"[1445652, 1270160, 980038]"
4,4,"[405663, 1524585, 479970, 521986, 496713, 1032...",13595840,"[1788508, 1788508, 1788508]"


In [21]:
%%time

preds = recs["item_id"].tolist()
preds = [list(pred) for pred in preds]

targets = recs["target"].tolist()
targets = [list(target) for target in targets]

utils.recall_several_targets(preds, targets)

CPU times: user 3.67 s, sys: 60.3 ms, total: 3.73 s
Wall time: 3.72 s


0.043986157571296085

In [22]:
stop

NameError: name 'stop' is not defined

In [ ]:
X_val.to_pandas()[["user", "target"]]

In [ ]:
recs['target'] = 

In [ ]:
recs.filter(pl.col("index") == 1).head(20)

In [ ]:
stop

In [ ]:
%%time

rec_cooc = coocurence.CoocurenceRecommender(
    top_pairs=top_pairs, item_index=0, num_recs=config.num_recs, exclude_seen_items=False
)

data_to_predict = X_val_seq.to_pandas()[config.col_train_seq].tolist()
df_preds = rec_cooc.recommend(data_to_predict)

df_val_users = pl.from_pandas(X_val_seq.to_pandas()[[config.col_user]].reset_index())
df_val_preds = df_val_users.join(df_preds, on='index', how='inner')
df_val_preds = df_val_preds.with_columns(
    [
        pl.col('cooc_rank_0')
        .cast(pl.Int32, strict=True)
        .alias('cooc_rank_0'),
    ]
)

df_val_preds.write_parquet(config.cur_model_output_path + f'cur_cooc_df_val_preds_{config.num_recs}.parquet')

In [ ]:
del data_to_predict, df_preds, df_val_users, df_val_preds, X_val_seq, X_val
gc.collect()

In [ ]:
del top_pairs
gc.collect()

# TEST

In [ ]:
%%time

PATH_DATA = config.path_data
df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')

test_users = df_test[config.col_user].unique()
df = pl.concat([df_train, df_test])
df = df.sort(config.col_ts, reverse=True)

df = df.to_pandas()
df[config.col_num_action] = df.groupby(config.col_user)[config.col_aid].cumcount()
df = pl.from_pandas(df)

df = df.with_columns(
    [
        pl.col(config.col_num_action)
        .cast(pl.Int32, strict=True)
        .alias(config.col_num_action),
        pl.col(config.col_n_session)
        .cast(pl.Int32, strict=True)
        .alias(config.col_n_session),
    ]
)

del df_train, df_test
gc.collect()

df_inter2inter = (
    df.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

list_inters = list(df_inter2inter.to_pandas()[config.col_aid])

del df_inter2inter, df
gc.collect()

In [ ]:
%%time

top_pairs = coocurence.train_cooc_model(data=list_inters, max_candidates=config.num_recs, normalize=True, filtering_rare_keys_trs=20)

In [ ]:
%%time

PATH_DATA = config.path_data
df_test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")
df_test = df_test.sort(config.col_ts)
# df_test = df_test.to_pandas()
# df_test[config.col_aid] = ordinal_encoder.transform(
#     df_test[config.col_aid].values.reshape(-1, 1)
# ).astype(int)
# df_test = pl.from_pandas(df_test)
df_test = df_test.groupby([config.col_user]).agg(
    [
        pl.col(config.col_aid).list().alias(config.col_train_seq),
    ]
)

data_to_predict = df_test.to_pandas()[config.col_train_seq].tolist()
rec_cooc = coocurence.CoocurenceRecommender(
    top_pairs=top_pairs, item_index=0, num_recs=config.num_recs, exclude_seen_items=False
)
df_preds = rec_cooc.recommend(data_to_predict)

df_test_users = pl.from_pandas(df_test.to_pandas()[[config.col_user]].reset_index())
df_test_preds = df_test_users.join(df_preds, on='index', how='inner')
df_test_preds = df_test_preds.with_columns(
    [
        pl.col('cooc_rank_0')
        .cast(pl.Int32, strict=True)
        .alias('cooc_rank_0'),
    ]
)

df_test_preds.write_parquet(config.cur_model_output_path + f'cur_cooc_df_test_preds_{config.num_recs}.parquet')